# CareLoop — Part 3: risk model
Run after Part 1. The test split is reserved for final reporting.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv('appointments_clean.csv')
ids = df['appointment_id'].copy()
X = df[['age','distance_km','fee_inr','lead_days','prior_no_shows','reminder_sent','booking_channel','visit_type']].copy()
X['reminder_sent'] = X['reminder_sent'].map({'Yes':1,'No':0})
X = pd.get_dummies(X, columns=['booking_channel','visit_type'])
y = df['no_show']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print('Features:', X.shape[1], 'Train:', len(X_train), 'Test:', len(X_test), 'Test no-shows:', int(y_test.sum()))
print('Always Attended baseline accuracy:', accuracy_score(y_test, np.zeros(len(y_test),dtype=int)))


## 3.2 Logistic regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000).fit(X_train_s,y_train)
lr_prob = lr.predict_proba(X_test_s)[:,1]
lr_pred = (lr_prob >= 0.5).astype(int)
print('accuracy',accuracy_score(y_test,lr_pred),'confusion',confusion_matrix(y_test,lr_pred))
print('precision',precision_score(y_test,lr_pred),'recall',recall_score(y_test,lr_pred),'f1',f1_score(y_test,lr_pred),'auc',roc_auc_score(y_test,lr_prob))
print(pd.Series(lr.coef_[0],index=X.columns).sort_values())
reminder_coef = lr.coef_[0][list(X.columns).index('reminder_sent')]
print('exp(scaled coefficient):',np.exp(reminder_coef))
print('exp(No-to-Yes effect):',np.exp(reminder_coef/scaler.scale_[list(X.columns).index('reminder_sent')]))


## 3.3 Decision tree depth comparison

In [ ]:
for depth in [None,3]:
    tree=DecisionTreeClassifier(random_state=42,max_depth=depth).fit(X_train,y_train)
    print('depth',depth,'actual depth',tree.get_depth(),'train accuracy',tree.score(X_train,y_train),'test accuracy',tree.score(X_test,y_test))


## 3.4 Random forest

In [ ]:
rf=RandomForestClassifier(n_estimators=100,random_state=42).fit(X_train,y_train)
rf_prob=rf.predict_proba(X_test)[:,1]
print('accuracy',rf.score(X_test,y_test),'recall',recall_score(y_test,rf.predict(X_test)),'auc',roc_auc_score(y_test,rf_prob))
print(pd.Series(rf.feature_importances_,index=X.columns).sort_values(ascending=False).head(3))

# Reporting comparison only; final model choice uses training CV below.
tree3=DecisionTreeClassifier(random_state=42,max_depth=3).fit(X_train,y_train)
comparison=pd.DataFrame([
 {'model':'logistic regression','accuracy':accuracy_score(y_test,lr_pred),'recall':recall_score(y_test,lr_pred),'roc_auc':roc_auc_score(y_test,lr_prob)},
 {'model':'depth-3 tree','accuracy':tree3.score(X_test,y_test),'recall':recall_score(y_test,tree3.predict(X_test)),'roc_auc':roc_auc_score(y_test,tree3.predict_proba(X_test)[:,1])},
 {'model':'random forest','accuracy':rf.score(X_test,y_test),'recall':recall_score(y_test,rf.predict(X_test)),'roc_auc':roc_auc_score(y_test,rf_prob)}])
print(comparison.to_string(index=False))


## 3.5 Cross-validation, threshold selection and one final test evaluation

In [ ]:
lr_pipe=Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(max_iter=1000))])
tree3=DecisionTreeClassifier(random_state=42,max_depth=3)
rf_base=RandomForestClassifier(n_estimators=100,random_state=42)
grid=GridSearchCV(RandomForestClassifier(random_state=42),{'n_estimators':[100,200],'max_depth':[3,5,8]},cv=5,scoring='roc_auc').fit(X_train,y_train)
models={'logistic regression':lr_pipe,'depth-3 tree':tree3,'random forest':rf_base}
cv_scores={name:cross_val_score(model,X_train,y_train,cv=5,scoring='roc_auc').mean() for name,model in models.items()}
cv_scores['tuned forest']=grid.best_score_
print('Tuned forest params:',grid.best_params_,'CV AUC:',grid.best_score_)
print('Training CV AUC:',cv_scores)
# The highest training CV AUC selects the model; never inspect test data for this choice.
winner_name=max(cv_scores,key=cv_scores.get)
winner={'logistic regression':lr_pipe,'depth-3 tree':tree3,'random forest':rf_base,'tuned forest':grid.best_estimator_}[winner_name]
oof=cross_val_predict(winner,X_train,y_train,cv=5,method='predict_proba')[:,1]
for threshold in [0.5,0.35]:
    pred=(oof>=threshold).astype(int)
    print('threshold',threshold,'precision',precision_score(y_train,pred),'recall',recall_score(y_train,pred),'f1',f1_score(y_train,pred),'flagged',int(pred.sum()))
# Choose the threshold with higher out-of-fold F1 (tie -> higher recall).
chosen=max([0.5,0.35],key=lambda t:(f1_score(y_train,(oof>=t).astype(int)),recall_score(y_train,(oof>=t).astype(int))))
print('Winner:',winner_name,'threshold:',chosen)
winner.fit(X_train,y_train)
probs=winner.predict_proba(X_test)[:,1]
flags=(probs>=chosen).astype(int)
print('Final test precision',precision_score(y_test,flags),'recall',recall_score(y_test,flags),'flagged',int(flags.sum()),'ROC-AUC',roc_auc_score(y_test,probs))
pd.DataFrame({'appointment_id':ids.loc[X_test.index].values,'no_show':y_test.values,'risk_prob':probs,'flag':flags}).to_csv('test_risk_scores.csv',index=False)
